## 상품 CRUD - products

In [10]:
import os
from supabase import create_client
from dotenv import load_dotenv
load_dotenv()

True

In [11]:
url = os.getenv('API_URL2')
key = os.getenv('API_KEY2')

supabase = create_client(url, key)
print(supabase)
print(url, key)

In [12]:
email = 'yeji2@test.org'
password = 'testtest1234'

# 회원가입
sign_in = supabase.auth.sign_up({
    'email' : email,
    'password' : password
})

In [13]:
# 로그인
print(email, password)
log_in = supabase.auth.sign_in_with_password({
    "email": email,
    "password": password
})

log_in.user

yeji2@test.org testtest1234


User(id='f875c8a3-76b6-4edf-ad36-78960d502568', app_metadata={'provider': 'email', 'providers': ['email']}, user_metadata={'email': 'yeji2@test.org', 'email_verified': True, 'phone_verified': False, 'sub': 'f875c8a3-76b6-4edf-ad36-78960d502568'}, aud='authenticated', confirmation_sent_at=datetime.datetime(2026, 9, 10, 7, 54, 40, 86273, tzinfo=TzInfo(0)), recovery_sent_at=None, email_change_sent_at=None, new_email=None, new_phone=None, invited_at=None, action_link=None, email='yeji2@test.org', phone='', created_at=datetime.datetime(2026, 9, 10, 7, 54, 40, 65251, tzinfo=TzInfo(0)), confirmed_at=datetime.datetime(2026, 9, 10, 7, 57, 45, 340391, tzinfo=TzInfo(0)), email_confirmed_at=datetime.datetime(2026, 9, 10, 7, 57, 45, 340391, tzinfo=TzInfo(0)), phone_confirmed_at=None, last_sign_in_at=datetime.datetime(2026, 9, 10, 7, 57, 46, 395479, tzinfo=TzInfo(0)), role='authenticated', updated_at=datetime.datetime(2026, 9, 10, 7, 57, 46, 397506, tzinfo=TzInfo(0)), identities=[UserIdentity(id='f8

In [14]:
# 로그인 한 유저 확인
login_user = supabase.auth.get_user()

if login_user :
    print("로그인 상태 :", login_user)
else : 
    print("미로그인 상태")

로그인 상태 : user=User(id='f875c8a3-76b6-4edf-ad36-78960d502568', app_metadata={'provider': 'email', 'providers': ['email']}, user_metadata={'email': 'yeji2@test.org', 'email_verified': True, 'phone_verified': False, 'sub': 'f875c8a3-76b6-4edf-ad36-78960d502568'}, aud='authenticated', confirmation_sent_at=datetime.datetime(2026, 9, 10, 7, 54, 40, 86273, tzinfo=TzInfo(0)), recovery_sent_at=None, email_change_sent_at=None, new_email=None, new_phone=None, invited_at=None, action_link=None, email='yeji2@test.org', phone='', created_at=datetime.datetime(2026, 9, 10, 7, 54, 40, 65251, tzinfo=TzInfo(0)), confirmed_at=datetime.datetime(2026, 9, 10, 7, 57, 45, 340391, tzinfo=TzInfo(0)), email_confirmed_at=datetime.datetime(2026, 9, 10, 7, 57, 45, 340391, tzinfo=TzInfo(0)), phone_confirmed_at=None, last_sign_in_at=datetime.datetime(2026, 9, 10, 7, 57, 46, 395479, tzinfo=TzInfo(0)), role='authenticated', updated_at=datetime.datetime(2026, 9, 10, 7, 57, 46, 397506, tzinfo=TzInfo(0)), identities=[UserI

In [19]:
class product_service: # create, update, delete 등등
    
    def __init__ (self) : 
        user_info = supabase.auth.get_user()

        if not user_info : 
            raise PermissionError("로그인 후 다시 시도하세요")

        self.user = user_info.user


    #~~[create] - supabase table editor로 products 테이블 구조 만듦~~


    # [insert] - 신규 상품등록
    def new_product(self, name, price, description) :
        
        # 1. 현재 로그인한 사용자의 회원 유형 조회
        user_detail = (
            supabase.table("user_details")
                .select("type")
                .eq("id", self.user.id)
                .execute()
        )

        # 2. 회원 유형이 seller인 경우만 상품 등록할 수 있도록 
        if not user_detail.data : 
            raise PermissionError("회원가입을 먼저 진행해주세요")

        if user_detail.data["type"] != "seller" :
            raise PermissionError("Seller로 등록된 회원이 아닙니다")
        
        # 3. seller 회원 - 상품등록
        new_product = (
            supabase.table('products')
                .insert({
                    "name" : name,
                    "price" : price,
                    "description" : description,
                    "seller_id" : self.user.id
                })
                .execute()
        )
        return f"등록한 데이터 : {new_product.data}"

    
    # [select] - 상품 조회
    def get_product(self, name) :
        get_product = (
            supabase.table("products")
                .select("name", "price", "description")
                .ilke("name", f"%{name}%")
                .execute()
        )

        return f"조회된 데이터 : {get_product.data}"


    # [update] - 상품명 수정
    def modify_product_name(self, id, name) :
        responce_product_name = (
            supabase.table("products")
            .update({
                "name" : name
                })
                .eq("id", id)
                .eq("seller_id", self.user.id) # seller만 수정 가능하도록
                .execute()
        )

        return f"수정된 [[상품명]] 데이터 : {responce_product_name.data}"


    # [update] - 상품가격 수정
    def modify_product_price(self, id, price) : 
        response_product_price = (
            supabase.table("products")
            .update({
                "price" : price
            })
            .eq("id", id)
            .eq("seller_id", self.user.id) # seller만 수정 가능하도록
            .execute()
        )

        return f"수정된 [[가격]] 데이터 : {response_product_price.data}"


     # [update] - 상품 설명 수정
    def modify_product_description(self, id, description) :
        response_product_description = (
            supabase.table("products")
            .update({
                 "description" : description
            })
            .eq("id", id)
            .eq("seller_id", self.user.id) # seller만 수정 가능하도록
            .execute()
        )

        return f"수정된 [[상품설명]] 데이터 : {response_product_description.data}"